# Periphocal 3D interactive notebook

This notebook wraps the Python backend and adds an interactive workflow for:

- loading **CT DICOM** and **NIfTI segmentations**
- visually choosing a treatment **isocenter** on orthogonal CT views
- entering treatment parameters such as **field area, MU, Rx Gy or EU**
- computing **organ mean / min / max dose** and **DVHs**
- exporting the **full-body out-of-field dose cube** as **RTDOSE DICOM** and **NIfTI**
- interactively browsing the CT with a **dose overlay**
- inspecting organ DVHs with a **dropdown** so the notebook stays compact

## Important model caveat

This backend implements the **Periphocal 3D out-of-field model**. It is intended for **peripheral / out-of-field dose**, not as a replacement for TPS dose near or inside the treated region.

## Files expected next to this notebook

Place this notebook in the same folder as:

- `periphocal3d_dvh_ctdicom_nifti_rtdose.py`

The provided ZIP bundle already contains both files.


The notebook now shows only the common-path inputs by default; less common controls are collapsed under **Advanced** sections.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:

from pathlib import Path
import importlib.util
import sys
import json
import math
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown

# Optional, but the notebook also works without nibabel/pydicom imports here because the backend imports them.
import nibabel as nib
!pip install pydicom

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['image.cmap'] = 'gray'

BACKEND_PATH = Path.cwd() / 'periphocal3d_dvh_ctdicom_nifti_rtdose.py'
if not BACKEND_PATH.exists():
    raise FileNotFoundError(
        f'Cannot find backend script at {BACKEND_PATH}. Put the notebook next to {BACKEND_PATH.name}.'
    )

spec = importlib.util.spec_from_file_location('p3d_backend', BACKEND_PATH)
p3d = importlib.util.module_from_spec(spec)
sys.modules['p3d_backend'] = p3d
spec.loader.exec_module(p3d)

print(f'Loaded backend: {BACKEND_PATH}')
print('Backend functions available:', ', '.join(sorted([k for k in dir(p3d) if not k.startswith('_')])[:12]), '...')

BASE_DIR = Path.cwd()
DEFAULT_OUTDIR = BASE_DIR / 'Output'
DEFAULT_OUTDIR.mkdir(parents=True, exist_ok=True)
print(f'Default output folder: {DEFAULT_OUTDIR}')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 24.2 MB/s eta 0:00:00
Loaded backend: /content/periphocal3d_dvh_ctdicom_nifti_rtdose.py
Backend functions available: A1_MGY_CM2_PER_MU, A2_MGY_CM_PER_MU, A3_PER_CM, CtSeries, Dataset, DicomSequence, Dict, ExplicitVRLittleEndian, FileDataset, LPS_TO_RAS, List, ModelInputs ...
Default output folder: /content/Output


In [5]:

# -----------------------------
# Interactive notebook frontend
# -----------------------------
state = {
    'ct': None,
    'rois': [],
    'roi_map': {},
    'roi_mask_cache': {},
    'dose_cube': None,
    'summary_df': None,
    'dvh_map': {},
    'output_paths': {},
}


def sanitize_name(name: str) -> str:
    return p3d.sanitize_filename(name)


def parse_optional_float(text: str):
    text = str(text).strip()
    return None if text == '' else float(text)


def build_roi_mask(ct, roi):
    cache = state['roi_mask_cache']
    if roi.name in cache:
        return cache[roi.name]
    mask = np.zeros(ct.shape, dtype=bool)
    flat = mask.reshape(-1, order='C')
    flat[roi.flat_indices_c] = True
    cache[roi.name] = mask
    return mask


def get_plane(vol, plane, row, col, slc):
    if plane == 'axial':
        arr = vol[:, :, slc]
        cross_v = col
        cross_h = row
        xlabel, ylabel = 'col', 'row'
    elif plane == 'coronal':
        arr = vol[:, col, :]
        cross_v = slc
        cross_h = row
        xlabel, ylabel = 'slice', 'row'
    elif plane == 'sagittal':
        arr = vol[row, :, :]
        cross_v = slc
        cross_h = col
        xlabel, ylabel = 'slice', 'col'
    else:
        raise ValueError(f'Unknown plane: {plane}')
    return np.asarray(arr), cross_v, cross_h, xlabel, ylabel


def iso_mm_from_sliders():
    ct = state['ct']
    if ct is None:
        return None
    ijk = np.array([[int(iso_row.value), int(iso_col.value), int(iso_slice.value)]], dtype=float)
    return ct.index_to_patient_mm(ijk)[0]


def mm_to_sliders(xyz_mm):
    """Convert LPS mm coordinates to voxel indices and update sliders."""
    ct = state['ct']
    if ct is None:
        return
    ijk = ct.inverse_affine_lps @ np.array([xyz_mm[0], xyz_mm[1], xyz_mm[2], 1.0])
    r, c, s = int(round(ijk[0])), int(round(ijk[1])), int(round(ijk[2]))
    r = max(iso_row.min, min(iso_row.max, r))
    c = max(iso_col.min, min(iso_col.max, c))
    s = max(iso_slice.min, min(iso_slice.max, s))
    # Temporarily suppress observer callbacks to avoid loops
    iso_row.unobserve_all()
    iso_col.unobserve_all()
    iso_slice.unobserve_all()
    iso_row.value = r
    iso_col.value = c
    iso_slice.value = s
    # Re-attach observers
    for w in [iso_row, iso_col, iso_slice]:
        w.observe(render_view, names='value')
        w.observe(update_iso_label, names='value')
    update_iso_label()
    render_view()


def on_iso_voxel_changed(*_):
    """Called when user manually types voxel indices into the text boxes."""
    ct = state['ct']
    if ct is None:
        return
    try:
        r = int(iso_row_text.value)
        c = int(iso_col_text.value)
        s = int(iso_slice_text.value)
    except (ValueError, TypeError):
        return  # ignore incomplete input
    # Clamp to valid range
    r = max(iso_row.min, min(iso_row.max, r))
    c = max(iso_col.min, min(iso_col.max, c))
    s = max(iso_slice.min, min(iso_slice.max, s))
    # Temporarily detach all slider observers to avoid loops
    iso_row.unobserve_all()
    iso_col.unobserve_all()
    iso_slice.unobserve_all()
    iso_row.value = r
    iso_col.value = c
    iso_slice.value = s
    # Re-attach slider observers
    for w in [iso_row, iso_col, iso_slice]:
        w.observe(render_view, names='value')
        w.observe(update_iso_label, names='value')
    update_iso_label()
    render_view()


def on_iso_mm_changed(*_):
    """Called when user edits the mm text fields."""
    ct = state['ct']
    if ct is None:
        return
    try:
        x = float(iso_x_mm.value)
        y = float(iso_y_mm.value)
        z = float(iso_z_mm.value)
        mm_to_sliders([x, y, z])
    except (ValueError, TypeError):
        pass  # ignore incomplete input


def update_iso_label(*_):
    ct = state['ct']
    if ct is None:
        iso_label.value = '<b>Isocenter:</b> case not loaded'
        return
    xyz = iso_mm_from_sliders()
    iso_label.value = (
        f"<b>Isocenter voxel:</b> ({iso_row.value}, {iso_col.value}, {iso_slice.value})"
        f" &nbsp;&nbsp; <b>LPS mm:</b> ({xyz[0]:.2f}, {xyz[1]:.2f}, {xyz[2]:.2f})"
    )
    # Sync mm text fields (without triggering on_iso_mm_changed)
    iso_x_mm.unobserve_all()
    iso_y_mm.unobserve_all()
    iso_z_mm.unobserve_all()
    iso_x_mm.value = f'{xyz[0]:.2f}'
    iso_y_mm.value = f'{xyz[1]:.2f}'
    iso_z_mm.value = f'{xyz[2]:.2f}'
    iso_x_mm.observe(on_iso_mm_changed, names='value')
    iso_y_mm.observe(on_iso_mm_changed, names='value')
    iso_z_mm.observe(on_iso_mm_changed, names='value')
    # Sync voxel index text boxes without re-triggering their observers
    iso_row_text.unobserve_all()
    iso_col_text.unobserve_all()
    iso_slice_text.unobserve_all()
    iso_row_text.value = str(int(iso_row.value))
    iso_col_text.value = str(int(iso_col.value))
    iso_slice_text.value = str(int(iso_slice.value))
    iso_row_text.observe(on_iso_voxel_changed, names='value')
    iso_col_text.observe(on_iso_voxel_changed, names='value')
    iso_slice_text.observe(on_iso_voxel_changed, names='value')


def render_view(*_):
    ct = state['ct']
    with viewer_out:
        clear_output(wait=True)
        if ct is None:
            print('Load a case first.')
            return

        row = int(iso_row.value)
        col = int(iso_col.value)
        slc = int(iso_slice.value)
        organ_name = organ_overlay.value
        dose_cube = state.get('dose_cube') if show_dose.value else None
        dose_max = parse_optional_float(dose_display_max_gy.value)
        dose_alpha_value = float(dose_alpha.value)

        overlay_mask = None
        if organ_name != 'None' and organ_name in state['roi_map']:
            overlay_mask = build_roi_mask(ct, state['roi_map'][organ_name])

        fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
        for ax, plane in zip(axes, ['axial', 'coronal', 'sagittal']):
            img, cross_v, cross_h, xlabel, ylabel = get_plane(ct.volume_hu, plane, row, col, slc)
            ax.imshow(img, origin='lower', cmap='gray')

            if overlay_mask is not None:
                mask_plane, _, _, _, _ = get_plane(overlay_mask.astype(np.uint8), plane, row, col, slc)
                masked = np.ma.masked_where(mask_plane == 0, mask_plane)
                ax.imshow(masked, origin='lower', cmap='Greens', alpha=float(mask_alpha.value), interpolation='nearest')

            if dose_cube is not None:
                dose_plane, _, _, _, _ = get_plane(dose_cube, plane, row, col, slc)
                dose_masked = np.ma.masked_where(dose_plane <= 0, dose_plane)
                vmax = None
                if dose_max is not None and dose_max > 0:
                    vmax = dose_max
                else:
                    finite = dose_cube[np.isfinite(dose_cube)]
                    finite = finite[finite > 0]
                    vmax = float(np.percentile(finite, 99)) if finite.size else None
                ax.imshow(dose_masked, origin='lower', cmap='hot', alpha=dose_alpha_value, vmin=0.0, vmax=vmax)

            ax.axvline(cross_v, color='cyan', linewidth=1)
            ax.axhline(cross_h, color='cyan', linewidth=1)
            ax.set_title(plane.capitalize())
            ax.set_xlabel(xlabel)
            ax.set_ylabel(ylabel)

        plt.show()


def load_case_clicked(_=None):
    with log_out:
        clear_output()
        try:
            ct_dir_path = Path(ct_dir.value).expanduser()
            if not ct_dir_path.exists():
                raise FileNotFoundError(f'CT directory does not exist: {ct_dir_path}')

            ct = p3d.load_ct_series(ct_dir_path)

            if seg_nifti_dir.value.strip():
                seg = p3d.segmentation_from_nifti_dir(
                    Path(seg_nifti_dir.value).expanduser(),
                    ct,
                    seg_coord_system=seg_coord_system.value,
                    roi_regex=roi_regex.value.strip() or None,
                    min_voxels=int(min_voxels.value),
                )
            elif seg_nifti_file.value.strip():
                labelmap_json_path = Path(seg_labelmap_json.value).expanduser() if seg_labelmap_json.value.strip() else None
                seg = p3d.segmentation_from_single_nifti(
                    Path(seg_nifti_file.value).expanduser(),
                    ct,
                    seg_coord_system=seg_coord_system.value,
                    roi_regex=roi_regex.value.strip() or None,
                    min_voxels=int(min_voxels.value),
                    labelmap_json=labelmap_json_path,
                )
            else:
                raise ValueError('Provide either a segmentation directory or a segmentation NIfTI file.')

            state['ct'] = ct
            state['rois'] = seg.rois
            state['roi_map'] = {roi.name: roi for roi in seg.rois}
            state['roi_mask_cache'] = {}
            state['dose_cube'] = None
            state['summary_df'] = None
            state['dvh_map'] = {}
            state['output_paths'] = {}

            organ_names = ['None'] + [roi.name for roi in seg.rois]
            organ_overlay.options = organ_names
            organ_select.options = [roi.name for roi in seg.rois]
            if organ_names:
                organ_overlay.value = 'None'
            if seg.rois:
                organ_select.value = seg.rois[0].name

            rows, cols, slices = ct.shape
            iso_row.max = rows - 1
            iso_col.max = cols - 1
            iso_slice.max = slices - 1
            iso_row.value = rows // 2
            iso_col.value = cols // 2
            iso_slice.value = slices // 2

            update_iso_label()
            render_view()

            print(f'Loaded CT shape: {ct.shape}')
            print(f'Loaded {len(seg.rois)} ROIs')
            print('ROIs:')
            for roi in seg.rois:
                print(f'  - {roi.name} (nvox={roi.flat_indices_c.size})')

        except Exception as exc:
            print('Load failed:')
            traceback.print_exc()


def write_outputs_and_compute(_=None):
    with log_out:
        clear_output()
        ct = state['ct']
        if ct is None:
            print('Load a case first.')
            return

        try:
            outdir_path = DEFAULT_OUTDIR
            outdir.value = str(outdir_path)
            outdir_path.mkdir(parents=True, exist_ok=True)

            saved_ct_nifti = None
            if save_ct_nifti.value.strip():
                saved_ct_nifti = p3d.save_ct_as_nifti(ct, Path(save_ct_nifti.value).expanduser())
                print(f'Wrote CT NIfTI: {saved_ct_nifti}')

            total_mu_val = float(total_mu.value)
            rx_gy_val = parse_optional_float(rx_gy.value)
            eu_val = parse_optional_float(eu_mgy_per_mu.value)
            leakage_val = float(leakage_mgy_per_mu.value)

            model = p3d.ModelInputs(
                field_area_cm2=float(field_area_cm2.value),
                total_mu=total_mu_val,
                eu_mgy_per_mu=p3d.derive_eu_mgy_per_mu(rx_gy_val, total_mu_val, eu_val),
                leakage_mgy_per_mu=leakage_val,
            )
            iso_mm = iso_mm_from_sliders()
            rois = state['rois']

            summary_rows = []
            dvh_map = {}
            for roi in rois:
                dose_gy = p3d.evaluate_roi_dose_values_gy(ct, roi, iso_mm, model)
                finite = dose_gy[np.isfinite(dose_gy)]
                mean_gy = float(np.mean(finite)) if finite.size else math.nan
                min_gy = float(np.min(finite)) if finite.size else math.nan
                max_gy = float(np.max(finite)) if finite.size else math.nan

                dvh_df = p3d.cumulative_dvh(dose_gy, nbins=int(dvh_bins.value))
                base = sanitize_name(roi.name)
                dvh_csv_path = outdir_path / f'{base}_DVH.csv'
                dvh_png_path = outdir_path / f'{base}_DVH.png'
                dvh_df.to_csv(dvh_csv_path, index=False)
                p3d.write_dvh_plot(dvh_df, roi.name, dvh_png_path)
                dvh_map[roi.name] = {
                    'dose_values_gy': dose_gy,
                    'dvh_df': dvh_df,
                    'dvh_csv': dvh_csv_path,
                    'dvh_png': dvh_png_path,
                    'mean_gy': mean_gy,
                    'min_gy': min_gy,
                    'max_gy': max_gy,
                    'volume_cc': float(roi.flat_indices_c.size * ct.voxel_volume_cc),
                }
                summary_rows.append({
                    'roi_number': roi.number,
                    'roi_name': roi.name,
                    'roi_type': roi.roi_type,
                    'n_voxels': int(roi.flat_indices_c.size),
                    'voxel_volume_cc': ct.voxel_volume_cc,
                    'organ_volume_cc': float(roi.flat_indices_c.size * ct.voxel_volume_cc),
                    'dose_mean_Gy': mean_gy,
                    'dose_min_Gy': min_gy,
                    'dose_max_Gy': max_gy,
                    'dvh_csv': dvh_csv_path.name,
                    'dvh_png': dvh_png_path.name,
                })
                print(f'Processed ROI: {roi.name}')

            summary_df = pd.DataFrame(summary_rows)
            if not summary_df.empty:
                _sort_cols = [c for c in ['roi_number', 'roi_name'] if c in summary_df.columns]
                if _sort_cols:
                    summary_df = summary_df.sort_values(_sort_cols)
            summary_csv_path = outdir_path / 'organ_dose_summary.csv'
            summary_df.to_csv(summary_csv_path, index=False)
            print(f'Wrote summary: {summary_csv_path}')

            dose_cube = None
            full_nifti_path = None
            full_dicom_path = None
            full_npy_path = None
            if export_full_dose_cube.value:
                body_threshold = parse_optional_float(dose_cube_body_threshold_hu.value)
                dose_cube = p3d.compute_full_body_dose_cube_gy(
                    ct=ct,
                    isocenter_mm_lps=np.asarray(iso_mm, dtype=float),
                    model=model,
                    out_path=None,
                    chunk_slices=int(dose_cube_chunk_slices.value),
                    body_hu_threshold=body_threshold,
                )
                full_nifti_path = Path(full_dose_cube_nifti.value).expanduser() if full_dose_cube_nifti.value.strip() else (outdir_path / 'full_body_dose_Gy.nii.gz')
                full_dicom_path = Path(full_dose_cube_dicom.value).expanduser() if full_dose_cube_dicom.value.strip() else (outdir_path / 'full_body_dose_Gy.dcm')
                nib.save(p3d.dose_cube_nifti_image(ct, dose_cube), str(full_nifti_path))
                p3d.save_dose_cube_as_rtdose_dicom(ct, dose_cube.astype(np.float32), full_dicom_path)
                if full_dose_cube_npy.value.strip():
                    full_npy_path = Path(full_dose_cube_npy.value).expanduser()
                    full_npy_path.parent.mkdir(parents=True, exist_ok=True)
                    np.save(str(full_npy_path), dose_cube.astype(np.float32))
                print(f'Wrote full dose NIfTI: {full_nifti_path}')
                print(f'Wrote full dose RTDOSE DICOM: {full_dicom_path}')
                if full_npy_path is not None:
                    print(f'Wrote full dose NPY: {full_npy_path}')

            if export_legacy_mat.value:
                legacy_mat_path = outdir_path / 'phantom_p3d_python.mat'
                p3d.export_legacy_mat(legacy_mat_path, ct, rois)
                print(f'Wrote legacy MAT: {legacy_mat_path}')

            run_info = {
                'ct_dir': str(Path(ct_dir.value).expanduser()),
                'converted_ct_nifti': str(saved_ct_nifti) if saved_ct_nifti is not None else None,
                'segmentation_source_dir': seg_nifti_dir.value.strip() or None,
                'segmentation_source_file': seg_nifti_file.value.strip() or None,
                'seg_coord_system': seg_coord_system.value,
                'ct_shape': list(ct.shape),
                'ct_spacing_mm': [ct.row_spacing_mm, ct.col_spacing_mm, ct.slice_spacing_mm],
                'isocenter_voxel': [int(iso_row.value), int(iso_col.value), int(iso_slice.value)],
                'isocenter_mm_lps': [float(x) for x in iso_mm],
                'model_inputs': {
                    'field_area_cm2': model.field_area_cm2,
                    'total_mu': model.total_mu,
                    'eu_mgy_per_mu': model.eu_mgy_per_mu,
                    'epsilon': model.epsilon,
                    'field_factor': model.field_factor,
                    'leakage_mgy_per_mu': model.leakage_mgy_per_mu,
                },
                'full_body_dose_cube': {
                    'exported': bool(export_full_dose_cube.value),
                    'nifti_path': str(full_nifti_path) if full_nifti_path is not None else None,
                    'dicom_rtdose_path': str(full_dicom_path) if full_dicom_path is not None else None,
                    'npy_path': str(full_npy_path) if full_npy_path is not None else None,
                },
                'notes': [
                    'Periphocal 3D is an out-of-field model.',
                    'Dose inside or near the treatment field is outside the intended validity of the model.',
                    'The interactive viewer is slice-based, using axial/coronal/sagittal views.',
                ],
            }
            run_info_path = outdir_path / 'run_info.json'
            run_info_path.write_text(json.dumps(run_info, indent=2), encoding='utf-8')
            print(f'Wrote run info: {run_info_path}')

            state['dose_cube'] = dose_cube
            state['summary_df'] = summary_df
            state['dvh_map'] = dvh_map
            state['output_paths'] = {
                'summary_csv': summary_csv_path,
                'run_info_json': run_info_path,
                'full_nifti': full_nifti_path,
                'full_dicom': full_dicom_path,
                'full_npy': full_npy_path,
            }

            if len(organ_select.options) > 0:
                organ_select.value = organ_select.options[0]
            render_view()
            update_summary_table()
            update_organ_plot()
            print('Done.')

        except Exception:
            print('Dose calculation failed:')
            traceback.print_exc()


def update_summary_table(*_):
    with summary_out:
        clear_output(wait=True)
        summary_df = state.get('summary_df')
        if summary_df is None or summary_df.empty:
            print('No summary yet. Run the calculation first.')
            return
        display(summary_df[['roi_name', 'organ_volume_cc', 'dose_mean_Gy', 'dose_min_Gy', 'dose_max_Gy']])


def update_organ_plot(*_):
    with organ_plot_out:
        clear_output(wait=True)
        dvh_map = state.get('dvh_map', {})
        if not dvh_map:
            print('No DVH results yet. Run the calculation first.')
            return
        organ = organ_select.value
        if organ not in dvh_map:
            print('Select an organ.')
            return

        entry = dvh_map[organ]
        stats_html.value = (
            f"<b>{organ}</b><br>"
            f"Volume: {entry['volume_cc']:.2f} cc<br>"
            f"Mean: {entry['mean_gy']:.4f} Gy<br>"
            f"Min: {entry['min_gy']:.4f} Gy<br>"
            f"Max: {entry['max_gy']:.4f} Gy"
        )

        fig, ax = plt.subplots(1, 1, figsize=(6, 4), constrained_layout=True)
        df = entry['dvh_df']
        ax.plot(df['dose_Gy'], df['volume_percent'], linewidth=2)
        ax.set_title(f'DVH - {organ}')
        ax.set_xlabel('Dose (Gy)')
        ax.set_ylabel('Volume ≥ dose (%)')
        ax.grid(True, alpha=0.3)
        plt.show()


# -----------------
# Widget definition
# -----------------
ct_dir = widgets.Text(description='CT DICOM', placeholder='/path/to/ct_dir', layout=widgets.Layout(width='95%'))
seg_nifti_dir = widgets.Text(description='Seg dir', placeholder='/path/to/segmentation_dir', layout=widgets.Layout(width='95%'))
seg_nifti_file = widgets.Text(description='Seg file', placeholder='/path/to/segmentation.nii.gz', layout=widgets.Layout(width='95%'))
seg_labelmap_json = widgets.Text(description='Label JSON', placeholder='/path/to/labels.json', layout=widgets.Layout(width='95%'))
seg_coord_system = widgets.Dropdown(description='Seg coords', options=['ras', 'lps'], value='ras')
roi_regex = widgets.Text(description='ROI regex', placeholder='optional filter')
min_voxels = widgets.IntText(description='Min voxels', value=5)
load_case_btn = widgets.Button(description='Load CT + segments', button_style='primary')

iso_row = widgets.IntSlider(description='Iso row', min=0, max=1, step=1, value=0, continuous_update=False)
iso_col = widgets.IntSlider(description='Iso col', min=0, max=1, step=1, value=0, continuous_update=False)
iso_slice = widgets.IntSlider(description='Iso slice', min=0, max=1, step=1, value=0, continuous_update=False)
iso_x_mm = widgets.Text(description='Iso X (mm)', value='', placeholder='LPS x', layout=widgets.Layout(width='200px'))
iso_y_mm = widgets.Text(description='Iso Y (mm)', value='', placeholder='LPS y', layout=widgets.Layout(width='200px'))
iso_z_mm = widgets.Text(description='Iso Z (mm)', value='', placeholder='LPS z', layout=widgets.Layout(width='200px'))
iso_x_mm.observe(on_iso_mm_changed, names='value')
iso_y_mm.observe(on_iso_mm_changed, names='value')
iso_z_mm.observe(on_iso_mm_changed, names='value')
iso_row_text = widgets.Text(description='Iso row idx', value='0', placeholder='row index',
                            layout=widgets.Layout(width='175px'))
iso_col_text = widgets.Text(description='Iso col idx', value='0', placeholder='col index',
                            layout=widgets.Layout(width='175px'))
iso_slice_text = widgets.Text(description='Iso slc idx', value='0', placeholder='slice index',
                              layout=widgets.Layout(width='175px'))
iso_row_text.observe(on_iso_voxel_changed, names='value')
iso_col_text.observe(on_iso_voxel_changed, names='value')
iso_slice_text.observe(on_iso_voxel_changed, names='value')
iso_label = widgets.HTML(value='<b>Isocenter:</b> case not loaded')
organ_overlay = widgets.Dropdown(description='ROI overlay', options=['None'], value='None')
mask_alpha = widgets.FloatSlider(description='ROI alpha', min=0.0, max=1.0, step=0.05, value=0.30, continuous_update=False)
show_dose = widgets.Checkbox(description='Show dose', value=False)
dose_alpha = widgets.FloatSlider(description='Dose alpha', min=0.0, max=1.0, step=0.05, value=0.45, continuous_update=False)
dose_display_max_gy = widgets.Text(description='Dose max Gy', placeholder='auto (99th pct)')

field_area_cm2 = widgets.FloatText(description='Field cm²', value=50.0)
total_mu = widgets.FloatText(description='Total MU', value=500.0)
rx_gy = widgets.Text(description='Rx Gy', value='1.8')
eu_mgy_per_mu = widgets.Text(description='EU mGy/MU', value='')
leakage_mgy_per_mu = widgets.FloatText(description='Leak mGy/MU', value=float(p3d.REFERENCE_LEAKAGE_MGY_PER_MU))
dvh_bins = widgets.IntText(description='DVH bins', value=200)
outdir = widgets.Text(description='Out dir', value=str(DEFAULT_OUTDIR), layout=widgets.Layout(width='95%'))
outdir.disabled = True
save_ct_nifti = widgets.Text(description='Save CT NIfTI', placeholder='optional /path/to/ct_from_dicom.nii.gz', layout=widgets.Layout(width='95%'))
export_full_dose_cube = widgets.Checkbox(description='Export full cube', value=True)
full_dose_cube_nifti = widgets.Text(description='Dose NIfTI', placeholder='optional /path/to/full_body_dose_Gy.nii.gz', layout=widgets.Layout(width='95%'))
full_dose_cube_dicom = widgets.Text(description='Dose RTDOSE', placeholder='optional /path/to/full_body_dose_Gy.dcm', layout=widgets.Layout(width='95%'))
full_dose_cube_npy = widgets.Text(description='Dose NPY', placeholder='optional /path/to/full_body_dose_Gy.npy', layout=widgets.Layout(width='95%'))
dose_cube_chunk_slices = widgets.IntText(description='Chunk slices', value=8)
dose_cube_body_threshold_hu = widgets.Text(description='Body HU thr', value='')
export_legacy_mat = widgets.Checkbox(description='Export MAT', value=False)
run_btn = widgets.Button(description='Run dose + DVHs', button_style='success')

organ_select = widgets.Dropdown(description='Organ', options=[])
stats_html = widgets.HTML(value='')

viewer_out = widgets.Output()
summary_out = widgets.Output()
organ_plot_out = widgets.Output()
log_out = widgets.Output(layout=widgets.Layout(border='1px solid #ccc', max_height='280px', overflow='auto'))

load_case_btn.on_click(load_case_clicked)
run_btn.on_click(write_outputs_and_compute)
for w in [iso_row, iso_col, iso_slice, organ_overlay, mask_alpha, show_dose, dose_alpha, dose_display_max_gy]:
    w.observe(render_view, names='value')
for w in [iso_row, iso_col, iso_slice]:
    w.observe(update_iso_label, names='value')
organ_select.observe(update_organ_plot, names='value')


# -----------------------
# Layout: simple + advanced
# -----------------------
main_input_help = widgets.HTML(
    '<b>Usual case:</b> set <code>CT DICOM</code> and <code>Seg dir</code> '
    '(one NIfTI mask per organ), then click <b>Load CT + segments</b>.'
)

advanced_seg_box = widgets.VBox([
    widgets.HTML('<b>Advanced segmentation options</b>'),
    seg_nifti_file,
    seg_labelmap_json,
    widgets.HBox([seg_coord_system, roi_regex, min_voxels]),
])
advanced_seg_accordion = widgets.Accordion(children=[advanced_seg_box])
advanced_seg_accordion.set_title(0, 'Advanced segmentation options')
advanced_seg_accordion.selected_index = None

paths_box = widgets.VBox([
    widgets.HTML('<h3>1) Inputs</h3>'),
    main_input_help,
    ct_dir,
    seg_nifti_dir,
    advanced_seg_accordion,
    load_case_btn,
])

viewer_help = widgets.HTML(
    'Move the sliders to place the treatment isocenter. Overlay controls are optional.'
)
viewer_overlay_box = widgets.VBox([
    widgets.HBox([organ_overlay, mask_alpha]),
    widgets.HBox([show_dose, dose_alpha]),
    dose_display_max_gy,
])
viewer_overlay_accordion = widgets.Accordion(children=[viewer_overlay_box])
viewer_overlay_accordion.set_title(0, 'Advanced viewer options')
viewer_overlay_accordion.selected_index = None

viewer_box = widgets.VBox([
    widgets.HTML('<h3>2) Visual isocenter placement</h3>'),
    viewer_help,
    iso_label,
    iso_row, iso_col, iso_slice,
    widgets.HTML('<b>Or enter voxel indices directly:</b>'),
    widgets.HBox([iso_row_text, iso_col_text, iso_slice_text]),
    widgets.HTML('<b>Or enter coordinates directly (LPS mm):</b>'),
    widgets.HBox([iso_x_mm, iso_y_mm, iso_z_mm]),
    viewer_overlay_accordion,
    viewer_out,
])

main_param_help = widgets.HTML(
    'Enter the essentials: <code>Field cm²</code>, <code>Total MU</code>, and '
    '<code>Rx Gy</code>. Outputs are always written to <code>./Output</code> next to the notebook.'
)

advanced_run_box = widgets.VBox([
    widgets.HTML('<b>Advanced calculation/export options</b>'),
    widgets.HBox([leakage_mgy_per_mu, eu_mgy_per_mu, dvh_bins]),
    save_ct_nifti,
    widgets.HBox([export_full_dose_cube, dose_cube_chunk_slices, export_legacy_mat]),
    widgets.HBox([full_dose_cube_nifti, full_dose_cube_dicom]),
    widgets.HBox([full_dose_cube_npy, dose_cube_body_threshold_hu]),
])
advanced_run_accordion = widgets.Accordion(children=[advanced_run_box])
advanced_run_accordion.set_title(0, 'Advanced calculation/export options')
advanced_run_accordion.selected_index = None

params_box = widgets.VBox([
    widgets.HTML('<h3>3) Treatment parameters</h3>'),
    main_param_help,
    widgets.HBox([field_area_cm2, total_mu, rx_gy]),
    widgets.HTML(f'<i>Output folder:</i> <code>{DEFAULT_OUTDIR}</code>'),
    advanced_run_accordion,
    run_btn,
])

results_box = widgets.VBox([
    widgets.HTML('<h3>4) Results</h3>'),
    summary_out,
    widgets.HBox([organ_select, stats_html]),
    organ_plot_out,
])

display(paths_box)
display(viewer_box)
display(params_box)
display(results_box)
display(widgets.HTML('<h3>Log</h3>'))
display(log_out)

update_iso_label()
update_summary_table()
update_organ_plot()


HTML(value='<h3>Log</h3>')

Output(layout=Layout(border='1px solid #ccc', max_height='280px', overflow='auto'))

## Typical usage

1. Fill in **CT DICOM** and **Seg dir**.
2. Click **Load CT + segments**.
3. Move **Iso row / Iso col / Iso slice** to place the treatment isocenter.
4. Enter **Field cm²**, **Total MU**, **Rx Gy**, and **Out dir**.
5. Click **Run dose + DVHs**.
6. Open **Advanced** sections only if you need custom segmentation handling, direct EU input, or custom export paths.

## Notes

- If you already have one binary NIfTI file per organ, you can ignore **Seg file** and **Label JSON**.
- **Seg coords** should usually stay at `ras`.
- The dose viewer is a slice-based 3-plane viewer for checking overlays on the CT grid.
- The backend still assumes **out-of-field validity** for the Periphocal 3D dose model.
